# Canonical non-binary top-down oracle with Stanza trees

This notebook gives a compact implementation of canonical non-binary top-down parsing. The system uses no delimiters and does not encode arity in its reduction action. Every constituent label is introduced by `NT(L)`, its children are constructed from left to right, and the same `REDUCE` action constructs both leaf-parent and higher constituents.

The pipeline is parallel to the canonical bottom-up notebook:

1. read a PTB-style tree with Stanza,
2. extract a static top-down oracle,
3. apply the oracle to reconstruct a Stanza tree,
4. check exact tree reconstruction.

All example trees in this notebook are synthetic.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import List, Optional, Sequence, Tuple, Union

from stanza.models.constituency.parse_tree import Tree
from stanza.models.constituency.tree_reader import read_trees


# ============================================================
# Stanza tree utilities
# ============================================================

def bracket_diagnostics(ptb_string: str) -> dict:
    """Return a diagnostic report for PTB bracket balance."""
    balance = 0
    first_extra_close = None

    for line_no, line in enumerate(ptb_string.splitlines(), start=1):
        for col_no, char in enumerate(line, start=1):
            if char == "(":
                balance += 1
            elif char == ")":
                balance -= 1
                if balance < 0 and first_extra_close is None:
                    first_extra_close = (line_no, col_no)

    return {
        "open_parens": ptb_string.count("("),
        "close_parens": ptb_string.count(")"),
        "final_balance": balance,
        "first_extra_close": first_extra_close,
    }


def assert_balanced_ptb(ptb_string: str) -> None:
    """Raise a clear error if a PTB string has unmatched parentheses."""
    report = bracket_diagnostics(ptb_string)
    if report["first_extra_close"] is not None:
        line_no, col_no = report["first_extra_close"]
        raise ValueError(
            "Malformed PTB string: extra closing parenthesis at "
            f"line {line_no}, column {col_no}. "
            f"opens={report['open_parens']}, closes={report['close_parens']}."
        )
    if report["final_balance"] != 0:
        raise ValueError(
            "Malformed PTB string: unbalanced parentheses. "
            f"opens={report['open_parens']}, closes={report['close_parens']}, "
            f"final_balance={report['final_balance']}."
        )


def parse_ptb(ptb_string: str) -> Tree:
    """Read exactly one PTB-style bracketed tree with Stanza."""
    ptb_string = ptb_string.strip()
    assert_balanced_ptb(ptb_string)
    trees = list(read_trees(ptb_string, use_tqdm=False))
    if len(trees) != 1:
        raise ValueError(f"Expected exactly one tree, found {len(trees)}.")
    return trees[0]


def make_tree(label: str, children: Optional[Sequence[Tree]] = None) -> Tree:
    """Construct a Stanza constituency tree."""
    return Tree(label, [] if children is None else list(children))


def label(node: Tree) -> str:
    value = node.label
    return value() if callable(value) else value


def children(node: Tree) -> List[Tree]:
    value = node.children
    return list(value() if callable(value) else value)


def is_leaf(node: Tree) -> bool:
    return len(children(node)) == 0


def is_preterminal(node: Tree) -> bool:
    ch = children(node)
    return len(ch) == 1 and is_leaf(ch[0])


def terminal_of(preterminal: Tree) -> str:
    if not is_preterminal(preterminal):
        raise ValueError(f"Expected a leaf-parent constituent, found {to_ptb(preterminal)}")
    return label(children(preterminal)[0])


def yield_words(tree: Tree) -> List[str]:
    if is_leaf(tree):
        return [label(tree)]

    words: List[str] = []
    for child in children(tree):
        words.extend(yield_words(child))
    return words


def to_ptb(tree: Tree) -> str:
    """Return a compact PTB bracket string."""
    ch = children(tree)
    if not ch:
        return label(tree)
    return f"({label(tree)} {' '.join(to_ptb(child) for child in ch)})"


def pretty(tree: Tree) -> str:
    """Use Stanza's pretty printer when available, otherwise compact PTB."""
    if hasattr(tree, "pformat"):
        return tree.pformat()
    return to_ptb(tree)


def normalize_bracket_token(token: str) -> str:
    """Normalize literal bracket tokens to PTB bracket symbols."""
    return {
        "(": "-LRB-",
        ")": "-RRB-",
        "{": "-LCB-",
        "}": "-RCB-",
        "[": "-LSB-",
        "]": "-RSB-",
    }.get(token, token)


def assert_same_tree(left: Tree, right: Tree) -> None:
    left_ptb = to_ptb(left)
    right_ptb = to_ptb(right)
    if left_ptb != right_ptb:
        raise AssertionError(
            "Trees differ.\n\n"
            f"Gold:\n{left_ptb}\n\n"
            f"Pred:\n{right_ptb}"
        )

## Transition representation

The canonical top-down inventory contains label-specific `NT(L)`, `SHIFT`, `REDUCE`, and `FINISH`. The stack contains bare terminals, completed tree items, and unmatched label markers. `REDUCE` constructs `L(alpha_1, ..., alpha_k)` from the items above the topmost unmatched `NT(L)`.

In [ ]:
OracleOp = Tuple[str, Optional[str]]

SHIFT = "SHIFT"
NT = "NT"
REDUCE = "REDUCE"
FINISH = "FINISH"


@dataclass(frozen=True)
class LabelMarker:
    label: str

    def __repr__(self) -> str:
        return f"NT({self.label})"


StackItem = Union[Tree, LabelMarker]


def format_action(action: OracleOp) -> str:
    op, value = action
    if op in {SHIFT, NT}:
        return f"{op}({value})"
    return op


def print_oracle(oracle: Sequence[OracleOp]) -> None:
    for i, action in enumerate(oracle, 1):
        print(f"{i:3d}. {format_action(action)}")


def action_counts(oracle: Sequence[OracleOp]) -> dict:
    counts = {SHIFT: 0, NT: 0, REDUCE: 0, FINISH: 0}
    for op, _value in oracle:
        counts[op] = counts.get(op, 0) + 1
    return counts


def transition_inventory(oracle: Sequence[OracleOp]) -> List[str]:
    """Return distinct predicted transition types in an oracle."""
    inventory = {
        f"NT({value})" if op == NT else op
        for op, value in oracle
    }
    return sorted(inventory)

## Static oracle

For any constituent `L(C1, ..., Ck)`, the oracle emits `NT(L)`, constructs `C1, ..., Ck` from left to right, and emits `REDUCE`. Constructing a terminal child emits `SHIFT`. The reduction action contains neither the mother label nor the arity.

In [ ]:
def get_oracle_canonical_topdown(tree: Tree) -> List[OracleOp]:
    """Produce the canonical non-binary top-down static oracle."""
    oracle: List[OracleOp] = []

    def visit(node: Tree) -> None:
        if is_leaf(node):
            raise ValueError("A bare terminal must be visited through its parent constituent.")

        node_children = children(node)
        node_label = label(node)
        if not node_children:
            raise ValueError(f"Unexpected empty nonterminal: {node_label!r}")

        oracle.append((NT, node_label))

        if is_preterminal(node):
            oracle.append((SHIFT, terminal_of(node)))
        else:
            for child in node_children:
                visit(child)

        oracle.append((REDUCE, None))

    visit(tree)
    oracle.append((FINISH, None))
    return oracle

## Applying the oracle

The interpreter enforces the canonical top-down stack pattern uniformly for every `NT(L)`. `REDUCE` closes the topmost unmatched marker, combines the nonempty sequence of items above it, and assigns them the marker label. The interpreter also checks input consumption, singleton final stack, and the required start symbol.

In [ ]:
def apply_oracle_canonical_topdown(
    sentence: Sequence[str],
    oracle: Sequence[OracleOp],
    *,
    start_symbol: Optional[str] = None,
    verbose: bool = False,
) -> Tree:
    stack: List[StackItem] = []
    word_idx = 0
    finished = False
    final_tree: Optional[Tree] = None

    def item_string(item: StackItem) -> str:
        if isinstance(item, LabelMarker):
            return repr(item)
        return to_ptb(item)

    def stack_string() -> str:
        if not stack:
            return "[]"
        return "[ " + " | ".join(item_string(item) for item in stack) + " ]"

    def buffer_string() -> str:
        if word_idx >= len(sentence):
            return "[]"
        return "[ " + " ".join(sentence[word_idx:]) + " ]"

    def trace(step: int, action: OracleOp, when: str) -> None:
        if not verbose:
            return
        print(f"\nStep {step}: {format_action(action)} [{when}]")
        print(f"Buffer: {buffer_string()}")
        print(f"Stack : {stack_string()}")

    def do_nt(value: Optional[str]) -> None:
        if value is None:
            raise ValueError("NT requires a label.")
        if any(isinstance(item, Tree) and is_leaf(item) for item in stack):
            raise ValueError("NT is unavailable while a leaf-parent constituent is unfinished.")
        stack.append(LabelMarker(value))

    def do_shift(value: Optional[str]) -> None:
        nonlocal word_idx

        if word_idx >= len(sentence):
            raise ValueError("SHIFT exceeds sentence length.")
        if not stack or not isinstance(stack[-1], LabelMarker):
            raise ValueError(f"SHIFT requires an unmatched NT(L) at stack top: {stack_string()}")
        if any(isinstance(item, Tree) and is_leaf(item) for item in stack):
            raise ValueError("SHIFT is unavailable while a leaf-parent constituent is unfinished.")

        expected = normalize_bracket_token(sentence[word_idx])
        observed = normalize_bracket_token(value if value is not None else sentence[word_idx])
        if observed != expected:
            raise ValueError(
                f"SHIFT mismatch at position {word_idx}: "
                f"oracle has {observed!r}, sentence has {expected!r}."
            )

        stack.append(make_tree(expected))
        word_idx += 1

    def do_reduce() -> None:
        marker_positions = [
            index for index, item in enumerate(stack)
            if isinstance(item, LabelMarker)
        ]
        if not marker_positions:
            raise ValueError(f"REDUCE requires an unmatched marker: {stack_string()}")

        marker_index = marker_positions[-1]
        marker = stack[marker_index]
        assert isinstance(marker, LabelMarker)
        reduced_children = stack[marker_index + 1:]
        if not reduced_children or not all(isinstance(child, Tree) for child in reduced_children):
            raise ValueError(f"REDUCE requires a nonempty completed suffix: {stack_string()}")
        if any(is_leaf(child) for child in reduced_children):
            if len(reduced_children) != 1 or not is_leaf(reduced_children[0]):
                raise ValueError(f"A bare terminal must be the sole child of REDUCE: {stack_string()}")

        stack[:] = stack[:marker_index]
        stack.append(make_tree(marker.label, reduced_children))

    def do_finish() -> Tree:
        if word_idx != len(sentence):
            raise ValueError(
                f"FINISH before consuming all words: {word_idx}/{len(sentence)}"
            )
        if len(stack) != 1 or not isinstance(stack[0], Tree) or is_leaf(stack[0]):
            raise ValueError(
                f"FINISH requires exactly one completed tree: {stack_string()}"
            )
        if start_symbol is not None and label(stack[0]) != start_symbol:
            raise ValueError(
                f"FINISH requires root {start_symbol!r}, found {label(stack[0])!r}."
            )
        return stack[0]

    if verbose:
        print("=== APPLY CANONICAL NON-BINARY TOP-DOWN ORACLE ===")
        print("Sentence:", list(sentence))
        print("Oracle:")
        print_oracle(oracle)

    for step, action in enumerate(oracle, 1):
        op, value = action

        if finished:
            raise ValueError("No action may follow FINISH.")

        trace(step, action, "before")

        if op == NT:
            do_nt(value)
        elif op == SHIFT:
            do_shift(value)
        elif op == REDUCE:
            do_reduce()
        elif op == FINISH:
            final_tree = do_finish()
            finished = True
        else:
            raise ValueError(f"Unknown action: {op!r}")

        trace(step, action, "after")

    if not finished or final_tree is None:
        raise ValueError("Oracle ended without FINISH.")

    return final_tree

## End-to-end helper

In [ ]:
@dataclass
class CanonicalTopDownResult:
    gold_tree: Tree
    sentence: List[str]
    oracle: List[OracleOp]
    reconstructed_tree: Tree

    @property
    def gold_ptb(self) -> str:
        return to_ptb(self.gold_tree)

    @property
    def reconstructed_ptb(self) -> str:
        return to_ptb(self.reconstructed_tree)


def run_canonical_topdown_pipeline(
    ptb_string: str,
    verbose: bool = False,
    check: bool = True,
) -> CanonicalTopDownResult:
    gold_tree = parse_ptb(ptb_string)
    sentence = yield_words(gold_tree)
    oracle = get_oracle_canonical_topdown(gold_tree)
    reconstructed_tree = apply_oracle_canonical_topdown(
        sentence=sentence,
        oracle=oracle,
        start_symbol=label(gold_tree),
        verbose=verbose,
    )

    result = CanonicalTopDownResult(
        gold_tree=gold_tree,
        sentence=sentence,
        oracle=oracle,
        reconstructed_tree=reconstructed_tree,
    )

    if check:
        assert_same_tree(gold_tree, reconstructed_tree)

    return result

## Minimal example

In [ ]:
ptb_string = '(TOP (S (NP (NNP Mira) (NNP Vale)) (VP (VBZ builds) (NP (NN robots))) (. .)))'

result = run_canonical_topdown_pipeline(ptb_string, verbose=False)

print("Sentence:")
print(result.sentence)

print("\nOracle:")
print_oracle(result.oracle)

print("\nAction counts:")
print(action_counts(result.oracle))

print("\nReconstructed tree:")
print(result.reconstructed_ptb)

print("\nSame tree?")
print(result.gold_ptb == result.reconstructed_ptb)

## Larger example

In [ ]:
ptb_string = '(TOP (S (NP (DT The) (JJ young) (NN fox)) (VP (VBD watched) (SBAR (IN as) (S (NP (DT a) (NN bird)) (VP (VBD flew) (PP (IN over) (NP (DT the) (NN pond))))))) (. .)))'

result = run_canonical_topdown_pipeline(ptb_string, verbose=False)

print("Sentence length:", len(result.sentence))
print("Number of oracle actions:", len(result.oracle))
print("Action counts:", action_counts(result.oracle))
print("Distinct transition types:", len(transition_inventory(result.oracle)))
print("Same tree?", result.gold_ptb == result.reconstructed_ptb)
print("\nReconstructed tree:")
print(result.reconstructed_ptb)

## Regression checks

In [ ]:
examples = [
    '(TOP (S (NP (NNP Mira) (NNP Vale)) (VP (VBZ builds) (NP (NN robots))) (. .)))',
    '(TOP (S (NP (DT The) (JJ young) (NN fox)) (VP (VBD watched) (SBAR (IN as) (S (NP (DT a) (NN bird)) (VP (VBD flew) (PP (IN over) (NP (DT the) (NN pond))))))) (. .)))',
    '(TOP (S (VP (VB Wait))))',
    '(TOP (S (NP (NNP Ivo)) (VP (VBZ packs) (NP (NNS maps)) (NP (NNS snacks)) (PP (IN for) (NP (NN travel)))) (. .)))',
]

for i, example in enumerate(examples, 1):
    result = run_canonical_topdown_pipeline(example)
    assert result.gold_ptb == result.reconstructed_ptb
    print(
        f"Example {i}: OK, {len(result.oracle)} actions, "
        f"{len(transition_inventory(result.oracle))} distinct transition types"
    )

## Paper note

Canonical top-down parsing introduces every mother label as `NT(L)` before constructing its children. The generic `REDUCE` action contains neither the mother label nor the arity: the unmatched marker supplies the label, and its stack position identifies the ordered child sequence. Unlike delimiter-guided top-down parsing, the canonical system does not represent the left boundary with a separate delimiter.